Notebook structure:

First: Lesion graph from the SOOP dataset with usupervised anomaly maps from FLAIR and ADC maps with vascular territory information

Second: Same graph but with anomaly maps of healthy subjects (to look at the variability of anomaly maps on healthy vascular territories)

**Utiliser environment ordi local: base**


**Ou utiliser mon ordi perso avec Documents/general_env**

In [1]:
import argparse
import json
import csv
import glob

import os
import sys
sys.path.append("../..")
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np

import nibabel as nib

#import seaborn as sns
import matplotlib.pyplot as plt

#import ants

from multiprocessing import Pool
from functools import partial

from ipywidgets import Output, VBox
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import IntSlider, interact

### **Unsupervised anomaly map based multicontrast stroke lesion analysis**

This notebook gathers raw anomaly maps (no absolute value) from FLAIR and ADC images of large lesions from the SOOP dataset\
It combines the anomaly maps with an atlas for vascular territory information

In [2]:
#ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"
#ROOT_DIR = "/home/fehrdelt/bettik/"
#ROOT_DIR = "/home/rivage/bettik/"
ROOT_DIR = "/home/theotime/bettik/"

In [3]:
adc_folder = f"{ROOT_DIR}datasets/final_soop_dataset_small/adc_registered/"
flair_folder = f"{ROOT_DIR}datasets/final_soop_dataset_small/flair_registered/"

In [4]:
adc_anomaly_maps_folder_old = f"{ROOT_DIR}datasets/anomaly_maps_no_abs/exp_2_2/large/" # old normalize intensity by histogram peak on the whole batch at the same time
flair_anomaly_maps_folder_old = f"{ROOT_DIR}datasets/anomaly_maps_no_abs/exp_3_2/large/" # old normalize intensity by histogram peak on the whole batch at the same time

adc_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_2_2/large_no_abs_value/"
flair_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_3_2/large_no_abs_value/"

atlases_folder = f"{ROOT_DIR}datasets/final_soop_dataset_small/registered_atlases/"

In [9]:
unregistered_atlas = nib.load(f"{ROOT_DIR}datasets/registered_atlas_128.nii.gz").get_fdata()
atlas_zones_indexes = np.unique(unregistered_atlas)
atlas_zones_indexes = [int(idx) for idx in atlas_zones_indexes if idx != 0]
print("Atlas zones indexes:", atlas_zones_indexes)

Atlas zones indexes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 32]


--------------------------------------------

In [10]:
# Get all patient files
adc_files = glob.glob(f"{adc_anomaly_maps_folder}*.nii.gz")
patient_ids = [os.path.basename(f).replace('.nii.gz', '') for f in adc_files]

# Create list to store all data
all_data = []

for patient_id in tqdm(patient_ids):
    ano_adc = nib.load(f"{adc_anomaly_maps_folder}{patient_id}.nii.gz").get_fdata()
    ano_flair = nib.load(f"{flair_anomaly_maps_folder}{patient_id}.nii.gz").get_fdata()
    atlas = nib.load(f"{atlases_folder}{patient_id}.nii.gz").get_fdata().astype(int)
    
    for zone_idx in atlas_zones_indexes:
        zone_mask = (atlas == zone_idx)
        if np.sum(zone_mask) == 0:
            continue
        
        mean_adc = -np.mean(ano_adc[zone_mask]) # negative value to make it more understandable (low ADC = lower values)
        mean_flair = -np.mean(ano_flair[zone_mask])
        
        all_data.append({
            'patient_id': patient_id,
            'zone': zone_idx,
            'mean_adc': mean_adc,
            'mean_flair': mean_flair
        })

df_zones = pd.DataFrame(all_data)
df_zones

0it [00:00, ?it/s]


""


-----------------

In [7]:
atlas_zones_names = {
    "anterior cerebral artery left": 1,
    "anterior cerebral artery right": 2,
    "medial lenticulostriate left": 3,
    "medial lenticulostriate right": 4,
    "lateral lenticulostriate left": 5,
    "lateral lenticulostriate right": 6,
    "frontal pars of middle cerebral artery left": 7,
    "frontal pars of middle cerebral artery right": 8,
    "parietal pars of middle cerebral artery left": 9,
    "parietal pars of middle cerebral artery right": 10,
    "temporal pars of middle cerebral artery left": 11,
    "temporal pars of middle cerebral artery right": 12,
    "occipital pars of middle cerebral artery left": 13,
    "occipital pars of middle cerebral artery right": 14,
    "insular pars of middle cerebral artery left": 15,
    "insular pars of middle cerebral artery right": 16,
    "temporal pars of posterior cerebral artery left": 17,
    "temporal pars of posterior cerebral artery right": 18,
    "occipital pars of posterior cerebral artery left": 19,
    "occipital pars of posterior cerebral artery right": 20,
    "posterior choroidal and thalamoperfurators left": 21,
    "posterior choroidal and thalamoperfurators right": 22,
    "anteior choroidal and thalamoperfurators left": 23,
    "anterior choroidal and thalamoperfurators right": 24,
    "basilar left": 25,
    "basilar right": 26,
    "superior cerebellar left": 27,
    "superior cerebellar right": 28,
    "inferior cerebellar left": 29,
    "inferior cerebellar right": 30,
    "lateral ventricle left": 31,
    "lateral ventricle right": 32	
}

**enlever les petites zones de l'atlas psk ça fait des erreurs**

#### Unhealthy (SOOP large lesions)

In [17]:



fig = px.scatter(df_zones, x='mean_adc', y='mean_flair', 
                 color='patient_id', hover_data=['zone', 'patient_id'],
                 title='Mean ADC vs Mean FLAIR Values per Atlas Zone (All Patients)')

fig.update_layout(
    xaxis_title='Mean ADC Value',
    yaxis_title='Mean FLAIR Value',
    width=800, height=800
)

# Add reference lines
max_abs = max(df_zones['mean_adc'].abs().max(), df_zones['mean_flair'].abs().max())
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)

# Center the plot
fig.update_xaxes(range=[-max_abs * 1.1, max_abs * 1.1])
fig.update_yaxes(range=[-max_abs * 1.1, max_abs * 1.1])

# Add annotations for corners
fig.add_annotation(x=-max_abs, y=max_abs, text="Hypo ADC<br>Hyper FLAIR", showarrow=False, xanchor='left', yanchor='top')
fig.add_annotation(x=max_abs, y=max_abs, text="Hyper ADC<br>Hyper FLAIR", showarrow=False, xanchor='right', yanchor='top')
fig.add_annotation(x=-max_abs, y=-max_abs, text="Hypo ADC<br>Hypo FLAIR", showarrow=False, xanchor='left', yanchor='bottom')
fig.add_annotation(x=max_abs, y=-max_abs, text="Hyper ADC<br>Hypo FLAIR", showarrow=False, xanchor='right', yanchor='bottom')

fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.update_layout(legend_title_text='Patient ID', clickmode='event+select')


out = Output()

def on_click(trace, points, state):
    
    if points.point_inds:
        
        with out:
            out.clear_output(wait=True)
            idx = points.point_inds[0]
            
            patient_name = points.trace_name
            
            zone = df_zones.iloc[idx]['zone']
            
            # Get zone name from atlas_zones_names
            zone_name = [name for name, idx in atlas_zones_names.items() if idx == zone]
            zone_name = zone_name[0] if zone_name else f"Unknown zone {zone}"
        
        
            
            print(patient_id)
            print(f"Zone: {zone} - {zone_name}")
            
            ano_adc_img = nib.load(f"{adc_anomaly_maps_folder}{patient_name}.nii.gz").get_fdata()
            ano_flair_img = nib.load(f"{flair_anomaly_maps_folder}{patient_name}.nii.gz").get_fdata()
            adc_img = nib.load(f"{adc_folder}{patient_name}.nii.gz").get_fdata()
            flair_img = nib.load(f"{flair_folder}{patient_name}.nii.gz").get_fdata()
            atlas_img = nib.load(f"{atlases_folder}{patient_name}.nii.gz").get_fdata().astype(int)
            
            num_slices = ano_adc_img.shape[2]
            
            def update_slice(slice_idx):
                fig_img, axes = plt.subplots(2, 2, figsize=(12, 10))
                
                # Create zone contour for this slice
                zone_mask_slice = (atlas_img[:, :, slice_idx] == zone).astype(int)
                
                axes[0, 0].imshow(adc_img[:, :, slice_idx].T, origin='lower', cmap='gray')
                axes[0, 0].contour(zone_mask_slice.T, colors='red', linewidths=1)
                axes[0, 0].set_title(f'ADC - {patient_name} (slice {slice_idx})')
                axes[0, 0].axis('off')
                
                axes[0, 1].imshow(flair_img[:, :, slice_idx].T, origin='lower', cmap='gray')
                axes[0, 1].contour(zone_mask_slice.T, colors='red', linewidths=1)
                axes[0, 1].set_title(f'FLAIR - {patient_name} (slice {slice_idx})')
                axes[0, 1].axis('off')
                
                axes[1, 0].imshow(-ano_adc_img[:, :, slice_idx].T, origin='lower', cmap='bwr', vmin=-0.15, vmax=0.15)
                axes[1, 0].contour(zone_mask_slice.T, colors='black', linewidths=1)
                axes[1, 0].set_title(f'ADC Anomaly Map - {patient_name}')
                axes[1, 0].axis('off')
                
                axes[1, 1].imshow(-ano_flair_img[:, :, slice_idx].T, origin='lower', cmap='bwr', vmin=-0.15, vmax=0.15)
                axes[1, 1].contour(zone_mask_slice.T, colors='black', linewidths=1)
                axes[1, 1].set_title(f'FLAIR Anomaly Map - {patient_name}')
                axes[1, 1].axis('off')
                
                plt.suptitle(f'Patient: {patient_name}, Zone: {zone} - {zone_name}')
                plt.tight_layout()
                plt.show()
            
            slice_slider = IntSlider(min=0, max=num_slices-1, step=1, value=num_slices//2, description='Slice:')
            interact(update_slice, slice_idx=slice_slider)

fig_widget = go.FigureWidget(fig)
with fig_widget.batch_update():
    for trace in fig_widget.data:
        trace.on_click(on_click)

display(VBox([fig_widget, out]))

    'data': [{'customdata': array([[1, 'sub-631'],
                             …

#### **Same graph on "healthy" AINI-Stroke AIT images**

In [26]:
ait_adc_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_2_2/ait_no_abs/"
ait_flair_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_3_2/ait_no_abs/"

ait_adc_folder = f"{ROOT_DIR}datasets/aini-stroke_ait/adc_registered/"
ait_flair_folder = f"{ROOT_DIR}datasets/aini-stroke_ait/flair_registered/"

ait_atlases_folder = f"{ROOT_DIR}datasets/aini-stroke_ait/registered_atlases/"

view the atlases first to see if the registration worked ?

In [23]:
# Get all patient files
ait_adc_files = glob.glob(f"{ait_adc_anomaly_maps_folder}*.nii.gz")
ait_patient_ids = [os.path.basename(f).replace('.nii.gz', '') for f in ait_adc_files]

# Create list to store all data
all_data_ait = []

for patient_id in tqdm(ait_patient_ids):
    ano_adc = nib.load(f"{ait_adc_anomaly_maps_folder}{patient_id}.nii.gz").get_fdata()
    try:
        ano_flair = nib.load(f"{ait_flair_anomaly_maps_folder}{patient_id}.nii.gz").get_fdata()
    except FileNotFoundError:
        print(f"FLAIR anomaly map not found for patient {patient_id}, skipping patient.")
        continue

    atlas = nib.load(f"{ait_atlases_folder}{patient_id.replace("ano_map_", "")}.nii.gz").get_fdata().astype(int)
    
    for zone_idx in atlas_zones_indexes:
        zone_mask = (atlas == zone_idx)
        if np.sum(zone_mask) == 0:
            continue
        
        mean_adc = -np.mean(ano_adc[zone_mask]) # negative value to make it more understandable (low ADC = lower values)
        mean_flair = -np.mean(ano_flair[zone_mask])
        
        all_data_ait.append({
            'patient_id': patient_id,
            'zone': zone_idx,
            'mean_adc': mean_adc,
            'mean_flair': mean_flair
        })

df_zones_ait = pd.DataFrame(all_data_ait)
df_zones_ait

  6%|▋         | 2/32 [00:03<00:45,  1.51s/it]

FLAIR anomaly map not found for patient ano_map_aini-stroke_15092, skipping patient.


 19%|█▉        | 6/32 [00:13<00:55,  2.13s/it]

FLAIR anomaly map not found for patient ano_map_aini-stroke_9963, skipping patient.


 28%|██▊       | 9/32 [00:20<00:47,  2.07s/it]

...

 94%|█████████▍| 30/32 [01:11<00:04,  2.10s/it]

FLAIR anomaly map not found for patient ano_map_aini-stroke_23608, skipping patient.


100%|██████████| 32/32 [01:14<00:00,  2.33s/it]

FLAIR anomaly map not found for patient ano_map_aini-stroke_13151, skipping patient.


,patient_id,zone,mean_adc,mean_flair
0,ano_map_aini-stroke_21100,1,0.015553,0.011546
1,ano_map_aini-stroke_21100,2,0.011160,0.010680
2,ano_map_aini-stroke_21100,3,0.014434,0.001841
3,ano_map_aini-stroke_21100,4,0.014262,0.006460
4,ano_map_aini-stroke_21100,5,0.004097,0.013418
...,...,...,...,...
667,ano_map_aini-stroke_13542,27,0.015540,-0.026928
668,ano_map_aini-stroke_13542,28,0.006988,0.008755
669,ano_map_aini-stroke_13542,29,0.014069,-0.001855
670,ano_map_aini-stroke_13542,30,0.003365,0.041366


In [ ]:

fig = px.scatter(df_zones_ait, x='mean_adc', y='mean_flair', 
                 color='patient_id', hover_data=['zone', 'patient_id'],
                 title='Mean ADC vs Mean FLAIR Values per Atlas Zone (AINI-Stroke AIT patients)')

fig.update_layout(
    xaxis_title='Mean ADC Value',
    yaxis_title='Mean FLAIR Value',
    width=800, height=800
)

# Add reference lines
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)

# Set fixed plot range to -0.2 to 0.2
fig.update_xaxes(range=[-0.2, 0.2])
fig.update_yaxes(range=[-0.2, 0.2])

# Add annotations for corners
fig.add_annotation(x=-0.2, y=0.2, text="Hypo ADC<br>Hyper FLAIR", showarrow=False, xanchor='left', yanchor='top')
fig.add_annotation(x=0.2, y=0.2, text="Hyper ADC<br>Hyper FLAIR", showarrow=False, xanchor='right', yanchor='top')
fig.add_annotation(x=-0.2, y=-0.2, text="Hypo ADC<br>Hypo FLAIR", showarrow=False, xanchor='left', yanchor='bottom')
fig.add_annotation(x=0.2, y=-0.2, text="Hyper ADC<br>Hypo FLAIR", showarrow=False, xanchor='right', yanchor='bottom')

fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.update_layout(legend_title_text='Patient ID', clickmode='event+select')


out = Output()

def on_click(trace, points, state):
    
    if points.point_inds:
        
        with out:
            out.clear_output(wait=True)
            idx = points.point_inds[0]
            
            patient_name = points.trace_name
            
            zone = df_zones_ait.iloc[idx]['zone']
            
            # Get zone name from atlas_zones_names
            zone_name = [name for name, idx in atlas_zones_names.items() if idx == zone]
            zone_name = zone_name[0] if zone_name else f"Unknown zone {zone}"
        
        
            
            print(patient_id)
            print(f"Zone: {zone} - {zone_name}")
            
            ano_adc_img = nib.load(f"{ait_adc_anomaly_maps_folder}{patient_name}.nii.gz").get_fdata()
            ano_flair_img = nib.load(f"{ait_flair_anomaly_maps_folder}{patient_name}.nii.gz").get_fdata()
            adc_img = nib.load(f"{ait_adc_folder}{patient_name.replace("ano_map_", "")}.nii.gz").get_fdata()
            flair_img = nib.load(f"{ait_flair_folder}{patient_name.replace("ano_map_", "")}.nii.gz").get_fdata()
            atlas_img = nib.load(f"{ait_atlases_folder}{patient_name.replace("ano_map_", "")}.nii.gz").get_fdata().astype(int)
            
            num_slices = ano_adc_img.shape[2]
            
            def update_slice(slice_idx):
                fig_img, axes = plt.subplots(2, 2, figsize=(12, 10))
                
                # Create zone contour for this slice
                zone_mask_slice = (atlas_img[:, :, slice_idx] == zone).astype(int)
                
                axes[0, 0].imshow(adc_img[:, :, slice_idx].T, origin='lower', cmap='gray')
                axes[0, 0].contour(zone_mask_slice.T, colors='red', linewidths=1)
                axes[0, 0].set_title(f'ADC - {patient_name} (slice {slice_idx})')
                axes[0, 0].axis('off')
                
                axes[0, 1].imshow(flair_img[:, :, slice_idx].T, origin='lower', cmap='gray')
                axes[0, 1].contour(zone_mask_slice.T, colors='red', linewidths=1)
                axes[0, 1].set_title(f'FLAIR - {patient_name} (slice {slice_idx})')
                axes[0, 1].axis('off')
                
                axes[1, 0].imshow(-ano_adc_img[:, :, slice_idx].T, origin='lower', cmap='bwr', vmin=-0.15, vmax=0.15)
                axes[1, 0].contour(zone_mask_slice.T, colors='black', linewidths=1)
                axes[1, 0].set_title(f'ADC Anomaly Map - {patient_name}')
                axes[1, 0].axis('off')
                
                axes[1, 1].imshow(-ano_flair_img[:, :, slice_idx].T, origin='lower', cmap='bwr', vmin=-0.15, vmax=0.15)
                axes[1, 1].contour(zone_mask_slice.T, colors='black', linewidths=1)
                axes[1, 1].set_title(f'FLAIR Anomaly Map - {patient_name}')
                axes[1, 1].axis('off')
                
                plt.suptitle(f'Patient: {patient_name}, Zone: {zone} - {zone_name}')
                plt.tight_layout()
                plt.show()
            
            slice_slider = IntSlider(min=0, max=num_slices-1, step=1, value=num_slices//2, description='Slice:')
            interact(update_slice, slice_idx=slice_slider)

fig_widget = go.FigureWidget(fig)
with fig_widget.batch_update():
    for trace in fig_widget.data:
        trace.on_click(on_click)

display(VBox([fig_widget, out]))


    'data': [{'customdata': array([[1, 'ano_map_aini-stroke_21100'],
           …

#### **Healthy test set images**

Faut d'abord trouver un sous-jeu de données avec des patients qui ont a la fois adc et flair

-> Dallas Lifespan Brain Study

Si y'a écrit ses-wave dans le nom du fichier c'est que c'est Dallas

In [10]:
healthy_test_adc_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_2_2/healthy_test_set_no_abs_value/"
healthy_test_flair_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_3_2/healthy_test_set_no_abs_value/"

In [11]:

healthy_test_adc_files = glob.glob(f"{healthy_test_adc_anomaly_maps_folder}*.nii.gz")
healthy_test_flair_files = glob.glob(f"{healthy_test_flair_anomaly_maps_folder}*.nii.gz")

In [12]:
final_healthy_adc_files = []
final_healthy_flair_files = []

for healthy_adc_file in healthy_test_adc_files:
    
    if "ses-wave" in healthy_adc_file:
    
        patient_id = os.path.basename(healthy_adc_file).replace('_ADC.nii.gz', '')

        for healthy_flair_file in healthy_test_flair_files:
            if patient_id in healthy_flair_file:
                final_healthy_adc_files.append(healthy_adc_file)
                final_healthy_flair_files.append(healthy_flair_file)
                break

In [13]:
print(final_healthy_adc_files)
print(final_healthy_flair_files)

print(len(final_healthy_adc_files))
print(len(final_healthy_flair_files))

['/home/rivage/bettik/datasets/anomaly_maps/exp_2_2/healthy_test_set_no_abs_value/ano_map_sub-4540_ses-wave2_ADC.nii.gz', '/home/rivage/bettik/datasets/anomaly_maps/exp_2_2/healthy_test_set_no_abs_value/ano_map_sub-1665_ses-wave1_ADC.nii.gz', '/home/rivage/bettik/datasets/anomaly_maps/exp_2_2/healthy_test_set_no_abs_value/ano_map_sub-1139_ses-wave2_ADC.nii.gz']
['/home/rivage/bettik/datasets/anomaly_maps/exp_3_2/healthy_test_set_no_abs_value/ano_map_sub-4540_ses-wave2_acq-FLAIR_run-1_T2w.nii.gz', '/home/rivage/bettik/datasets/anomaly_maps/exp_3_2/healthy_test_set_no_abs_value/ano_map_sub-1665_ses-wave1_acq-FLAIR_run-1_T2w.nii.gz', '/home/rivage/bettik/datasets/anomaly_maps/exp_3_2/healthy_test_set_no_abs_value/ano_map_sub-1139_ses-wave2_acq-FLAIR_run-1_T2w.nii.gz']
3
3


#### Y'en a pas assez dans l'intersection, sinon faut prendre le train set ou le val set en plus

#### **Faut aussi recaler l'atlas sur ces images**

In [14]:
# atlas registration

def process_image_register(image_path, output_folder):
    #print(f"image path: {image_path}")
    #print(f"image basename: {os.path.basename(image_path)}")
    #print(f"output path: {os.path.join(output_folder, os.path.basename(image_path))}")
    t1w_image = ants.image_read(image_path)
    template_image = ants.image_read(ROOT_DIR+"datasets/mni_icbm152_nlin_sym_09a/128_mni_icbm152_t1_skullstrip.nii.gz")
    atlas_image = ants.image_read(ROOT_DIR+"datasets/registered_atlas_128.nii.gz")
    output_path = output_folder+f"/{os.path.basename(image_path)}"
    
    # if the output file already exists, skip
    if os.path.exists(output_path):
        return

    # Perform registration
    registration = ants.registration(fixed=t1w_image, moving=template_image, type_of_transform='antsRegistrationSyNsr') # https://antspy.readthedocs.io/en/latest/registration.html types of transforms

    # Apply the same transform to the atlas
    warped_atlas = ants.apply_transforms(fixed=t1w_image, moving=atlas_image, transformlist=registration['fwdtransforms'], interpolator='nearestNeighbor')
    ants.image_write(warped_atlas, output_path)

In [15]:
directory = ROOT_DIR+"datasets/healthy_ano_maps_analysis/"

In [16]:
"""for filepath in final_healthy_flair_files:
    process_image_register(filepath, directory)"""

'for filepath in final_healthy_flair_files:\n    process_image_register(filepath, directory)'

In [17]:
healthy_adc_ano_maps = []
healthy_flair_ano_maps = []
healthy_adc = []
healthy_flair = []
atlases = []

for file in os.listdir(directory):
    print(file)
    if "ADC" in file:
        healthy_adc.append(file)
    if "FLAIR" in file:
        healthy_flair.append(file)
    if "ano_map_ADC" in file:
        healthy_adc_ano_maps.append(file)
    if "ano_map_FLAIR" in file:
        healthy_flair_ano_maps.append(file)
    if "atlas" in file:
        atlases.append(file)

sub-4540_ses-wave2_atlas.nii.gz
sub-4540_ses-wave2_ADC.nii.gz
sub-1139_ses-wave2_ano_map_FLAIR.nii.gz
sub-1139_ses-wave2_ADC.nii.gz
sub-1665_ses-wave1_ano_map_ADC.nii.gz
sub-1139_ses-wave2_FLAIR.nii.gz
sub-1665_ses-wave1_atlas.nii.gz
sub-4540_ses-wave2_FLAIR.nii.gz
sub-1665_ses-wave1_ADC.nii.gz
sub-4540_ses-wave2_ano_map_FLAIR.nii.gz
sub-1665_ses-wave1_ano_map_FLAIR.nii.gz
sub-1139_ses-wave2_atlas.nii.gz
sub-1139_ses-wave2_ano_map_ADC.nii.gz
sub-4540_ses-wave2_ano_map_ADC.nii.gz
sub-1665_ses-wave1_FLAIR.nii.gz


In [18]:
print(healthy_adc_ano_maps)

['sub-1665_ses-wave1_ano_map_ADC.nii.gz', 'sub-1139_ses-wave2_ano_map_ADC.nii.gz', 'sub-4540_ses-wave2_ano_map_ADC.nii.gz']


In [19]:
# Create list to store all data
all_data_healthy = []

for file in tqdm(healthy_adc_ano_maps):
    name = os.path.basename(file)[:18]
    
    healthy_ano_adc = nib.load(f"{directory}/{name}_ano_map_ADC.nii.gz").get_fdata()
    healthy_ano_flair = nib.load(f"{directory}/{name}_ano_map_FLAIR.nii.gz").get_fdata()
    healthy_adc = nib.load(f"{directory}/{name}_ADC.nii.gz").get_fdata()
    healthy_flair = nib.load(f"{directory}/{name}_FLAIR.nii.gz").get_fdata()
    atlas = nib.load(f"{directory}/{name}_atlas.nii.gz").get_fdata().astype(int)
    
    for zone_idx in atlas_zones_indexes:
        zone_mask = (atlas == zone_idx)
        if np.sum(zone_mask) == 0:
            continue
        
        mean_adc = -np.mean(ano_adc[zone_mask]) # negative value to make it more understandable (low ADC = lower values)
        mean_flair = -np.mean(ano_flair[zone_mask])
        
        all_data_healthy.append({
            'patient_id': name,
            'zone': zone_idx,
            'mean_adc': mean_adc,
            'mean_flair': mean_flair
        })

df_zones_healthy = pd.DataFrame(all_data_healthy)
df_zones_healthy

100%|██████████| 3/3 [00:29<00:00,  9.93s/it]


,patient_id,zone,mean_adc,mean_flair
0,sub-1665_ses-wave1,1,-0.010816,0.005769
1,sub-1665_ses-wave1,2,-0.019663,0.006602
2,sub-1665_ses-wave1,3,-0.007029,0.004263
3,sub-1665_ses-wave1,4,-0.074293,0.018884
4,sub-1665_ses-wave1,5,0.014173,-0.014907
...,...,...,...,...
91,sub-4540_ses-wave2,27,-0.001696,-0.016580
92,sub-4540_ses-wave2,28,0.007977,-0.012917
93,sub-4540_ses-wave2,29,0.007102,-0.012703
94,sub-4540_ses-wave2,30,-0.001160,0.009131


In [ ]:



fig = px.scatter(df_zones_healthy, x='mean_adc', y='mean_flair', 
                 color='patient_id', hover_data=['zone', 'patient_id'],
                 title='Mean ADC vs Mean FLAIR Values per Atlas Zone (All Patients)')

fig.update_layout(
    xaxis_title='Mean ADC Value',
    yaxis_title='Mean FLAIR Value',
    width=800, height=800
)

# Add reference lines
max_abs = max(df_zones_healthy['mean_adc'].abs().max(), df_zones_healthy['mean_flair'].abs().max())
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)

# Center the plot
fig.update_xaxes(range=[-max_abs * 1.1, max_abs * 1.1])
fig.update_yaxes(range=[-max_abs * 1.1, max_abs * 1.1])

# Add annotations for corners
fig.add_annotation(x=-max_abs, y=max_abs, text="Hypo ADC<br>Hyper FLAIR", showarrow=False, xanchor='left', yanchor='top')
fig.add_annotation(x=max_abs, y=max_abs, text="Hyper ADC<br>Hyper FLAIR", showarrow=False, xanchor='right', yanchor='top')
fig.add_annotation(x=-max_abs, y=-max_abs, text="Hypo ADC<br>Hypo FLAIR", showarrow=False, xanchor='left', yanchor='bottom')
fig.add_annotation(x=max_abs, y=-max_abs, text="Hyper ADC<br>Hypo FLAIR", showarrow=False, xanchor='right', yanchor='bottom')

fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.update_layout(legend_title_text='Patient ID', clickmode='event+select')


out = Output()

def on_click(trace, points, state):
    
    if points.point_inds:
        
        with out:
            out.clear_output(wait=True)
            idx = points.point_inds[0]
            
            patient_name = points.trace_name
            
            zone = df_zones_healthy.iloc[idx]['zone']
            
            # Get zone name from atlas_zones_names
            zone_name = [name for name, idx in atlas_zones_names.items() if idx == zone]
            zone_name = zone_name[0] if zone_name else f"Unknown zone {zone}"
        
        
            
            print(patient_id)
            print(f"Zone: {zone} - {zone_name}")
            
            ano_adc_img = nib.load(f"{directory}{patient_name}_ano_map_ADC.nii.gz").get_fdata()
            ano_flair_img = nib.load(f"{directory}{patient_name}_ano_map_FLAIR.nii.gz").get_fdata()
            adc_img = nib.load(f"{directory}{patient_name}_ADC.nii.gz").get_fdata()
            flair_img = nib.load(f"{directory}{patient_name}_FLAIR.nii.gz").get_fdata()
            atlas_img = nib.load(f"{directory}{patient_name}_atlas.nii.gz").get_fdata().astype(int)
            
            num_slices = ano_adc_img.shape[2]
            
            def update_slice(slice_idx):
                fig_img, axes = plt.subplots(2, 2, figsize=(12, 10))
                
                # Create zone contour for this slice
                zone_mask_slice = (atlas_img[:, :, slice_idx] == zone).astype(int)
                
                axes[0, 0].imshow(adc_img[:, :, slice_idx].T, origin='lower', cmap='gray')
                axes[0, 0].contour(zone_mask_slice.T, colors='red', linewidths=1)
                axes[0, 0].set_title(f'ADC - {patient_name} (slice {slice_idx})')
                axes[0, 0].axis('off')
                
                axes[0, 1].imshow(flair_img[:, :, slice_idx].T, origin='lower', cmap='gray')
                axes[0, 1].contour(zone_mask_slice.T, colors='red', linewidths=1)
                axes[0, 1].set_title(f'FLAIR - {patient_name} (slice {slice_idx})')
                axes[0, 1].axis('off')
                
                axes[1, 0].imshow(-ano_adc_img[:, :, slice_idx].T, origin='lower', cmap='bwr', vmin=-0.15, vmax=0.15)
                axes[1, 0].contour(zone_mask_slice.T, colors='black', linewidths=1)
                axes[1, 0].set_title(f'ADC Anomaly Map - {patient_name}')
                axes[1, 0].axis('off')
                
                axes[1, 1].imshow(-ano_flair_img[:, :, slice_idx].T, origin='lower', cmap='bwr', vmin=-0.15, vmax=0.15)
                axes[1, 1].contour(zone_mask_slice.T, colors='black', linewidths=1)
                axes[1, 1].set_title(f'FLAIR Anomaly Map - {patient_name}')
                axes[1, 1].axis('off')
                
                plt.suptitle(f'Patient: {patient_name}, Zone: {zone} - {zone_name}')
                plt.tight_layout()
                plt.show()
            
            slice_slider = IntSlider(min=0, max=num_slices-1, step=1, value=num_slices//2, description='Slice:')
            interact(update_slice, slice_idx=slice_slider)
#fig.show()

fig_widget = go.FigureWidget(fig)
with fig_widget.batch_update():
    for trace in fig_widget.data:
        trace.on_click(on_click)

display(VBox([fig_widget, out]))

    'data': [{'customdata': array([[1, 'sub-1665_ses-wave1'],
                  …